# Preparing Your Model for Pretraining

## Objectives
By the end of this demo you will be able to:
1. **Configure** a LLaMA-style transformer architecture by setting key hyperparameters.
2. **Compare** three weight initialisation strategies used in real LLM pretraining pipelines.
3. **Inspect** model weights and count total trainable parameters.
4. **Run inference** on randomly initialised vs. pretrained models to observe the difference.
5. **Downscale** a pretrained model by removing layers (depth pruning).
6. **Save** the initialised model to Google Drive for use in the training loop.

## Description
Before training begins, you must decide *how to initialise your model's weights*.
This choice significantly impacts training stability, compute cost, and final quality.

This lab explores three practical strategies used in modern LLM pretraining pipelines,
using Meta's **LlamaConfig** architecture and HuggingFace's **SmolLM2-360M** —
a fully open-source, LLaMA-compatible model — as the reference pretrained checkpoint.

**Weight initialisation strategies covered:**

| Strategy | Use case |
|---|---|
| Random initialisation | Training a new model from scratch |
| Continued pretraining | Adapting an existing model to a new domain |
| Downscaling | Creating a smaller model from a larger pretrained one |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/Colab Notebooks/data"
print(f"Data directory: {data_dir}")


Mounted at /content/drive
Data directory: /content/drive/MyDrive/Colab Notebooks/data


In [2]:
# Uncomment if not already installed
!pip install transformers torch -q


In [3]:
import warnings
warnings.filterwarnings('ignore')

import torch

def fix_torch_seed(seed=42):
    """Fix all random seeds for full reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fix_torch_seed()
print(f"Seed fixed. PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")


Seed fixed. PyTorch version: 2.10.0+cpu
Device available: CPU


## 1. Model Configuration

We configure a **LLaMA-style** transformer — the same architecture family used in
Meta's LLaMA, Mistral, Qwen, and SmolLM2. HuggingFace's `LlamaConfig` lets us
define every architectural hyperparameter before any weights are created.

> The separation of *config* (architecture blueprint) from
> *weights* (learned values) is a key design principle in HuggingFace Transformers.
> The config defines the skeleton; weights fill it later — either randomly or
> from a pretrained checkpoint.


In [4]:
from transformers import LlamaConfig

# Inspect the default LlamaConfig (matches LLaMA-7B architecture)
config = LlamaConfig()
print(config)


LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 32000
}



### Customise for a Smaller Demo Model

We scale down the architecture to build a compact model suitable for running on a
single GPU (or CPU for demo purposes), while preserving the full LLaMA architectural
structure — the same design used in Meta's LLaMA, Mistral, and SmolLM2.

We use **SmolLM2-360M** by HuggingFace as our reference pretrained model throughout
this lab. It is:
- Fully open-source, no authentication required
- LLaMA-style architecture — directly compatible with `LlamaConfig` and `LlamaForCausalLM`
- Trained on high-quality data (FineWeb-Edu, The Stack) — produces coherent inference output

The table below shows the architectural choices and their rationale:

| Parameter | Full LLaMA-7B | Our Demo Config | What it controls |
|---|---|---|---|
| `num_hidden_layers` | 32 | 12 | Number of transformer blocks (depth) |
| `hidden_size` | 4096 | 960 | Width of the residual stream |
| `intermediate_size` | 11008 | 2560 | Width of the MLP feed-forward layer |
| `num_key_value_heads` | 32 | 5 | Grouped Query Attention (GQA) heads |
| `torch_dtype` | float32 | bfloat16 | Half-precision — halves GPU memory usage |
| `use_cache` | True | False | Must be False for gradient checkpointing |

> These values are not arbitrary — they must be **consistent**
> with the pretrained model you intend to load later (SmolLM2-360M).
> Always run `print(model.config)` after loading any pretrained model to verify
> the exact values before defining your custom config.


In [5]:
config.num_hidden_layers = 12       # reduced from 32
config.hidden_size = 960            # matches SmolLM2-360M
config.intermediate_size = 2560     # matches SmolLM2-360M MLP width
config.num_attention_heads = 15     # matches SmolLM2-360M
config.num_key_value_heads = 5      # Grouped Query Attention (GQA)
config.torch_dtype = "bfloat16"     # half-precision training
config.use_cache = False            # required for gradient checkpointing
print(config)


`torch_dtype` is deprecated! Use `dtype` instead!


LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 960,
  "initializer_range": 0.02,
  "intermediate_size": 2560,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 15,
  "num_hidden_layers": 12,
  "num_key_value_heads": 5,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.0.0",
  "use_cache": false,
  "vocab_size": 32000
}



## 2. Weight Initialisation Strategies

Weight initialisation is one of the most important decisions in the pretraining pipeline.
The three strategies below cover the full range from *training from scratch* to
*adapting an existing model*.


### 2a. Random Weight Initialisation

All weights are drawn from a **truncated normal distribution** (mean=0, std=0.02).
Values beyond ±2σ are clipped to 0.

**When to use:** You have large compute budget, massive data, and want a model
with no prior biases

> inference on a randomly initialised model produces **gibberish**.
> This concretely demonstrates *why* pretraining on large corpora is necessary
> before a model can generate meaningful text.


In [6]:
from transformers import LlamaForCausalLM

# Instantiate model with random weights using our custom config
model = LlamaForCausalLM(config)
print(model)


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 960)
    (layers): ModuleList(
      (0-11): 12 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=1920, bias=False)
          (k_proj): Linear(in_features=960, out_features=640, bias=False)
          (v_proj): Linear(in_features=960, out_features=640, bias=False)
          (o_proj): Linear(in_features=1920, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-06)
    (rotary_emb): L

In [7]:
def print_nparams(model):
    """Print total number of trainable parameters."""
    nparams = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {nparams:,}  (~{nparams/1e6:.0f}M)")

print_nparams(model)


Total parameters: 208,920,000  (~209M)


In [8]:
# Inspect raw random weights in the first attention layer's query projection
layer_name = "model.layers.0.self_attn.q_proj.weight"

for name, param in model.named_parameters():
    if name == layer_name:
        print(f"First 15 weights of '{layer_name}':")
        print(param.data.view(-1)[:15])
        break


First 15 weights of 'model.layers.0.self_attn.q_proj.weight':
tensor([ 0.0181, -0.0470,  0.0078,  0.0285,  0.0124,  0.0350,  0.0326,  0.0029,
        -0.0199, -0.0077,  0.0146,  0.0231,  0.0292, -0.0097, -0.0222])


#### Inference on a Randomly Initialised Model

Let's run inference **before any training** to see what pure random weights produce.
This is the baseline that motivates the entire pretraining process:


In [9]:
from transformers import AutoTokenizer, TextStreamer

# SmolLM2-360M tokenizer — LLaMA-compatible, 49152 vocab size
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

prompt = "Large language models are trained on"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f"Prompt: '{prompt}'")
print("\nModel output (random weights — expect gibberish):")
outputs = model.generate(
    **inputs,
    streamer=streamer,
    use_cache=True,
    max_new_tokens=64,
    do_sample=False
)


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Prompt: 'Large language models are trained on'

Model output (random weights — expect gibberish):
 airplane airplane airplaneSyn airplaneurableSynSynurableurableurableLAN singersLANLANLANLANLANLAN issueLAN issue singers issue singers issue singers issue singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers singers translation translation translation translation translation translation translation translation translation translation translation translation


In [10]:
# Free memory before loading the next model
import gc
del model, streamer, outputs
gc.collect()
print("Memory cleared.")


Memory cleared.


### 2b. Continued Pretraining (Reuse Pretrained Weights)

Load an already-pretrained model and **continue training** it on new data.
This is the most compute-efficient strategy when a strong general base model
already exists and you want to adapt it to a specific domain.

**When to use:**
- Adapting to a new domain (medical, legal, code, scientific)
- Adding a new language to an English-dominant model
- Updating a model with recent knowledge (temporal adaptation)

**Reference model:** `HuggingFaceTB/SmolLM2-360M`
- 360M parameters, LLaMA architecture
- Pretrained on FineWeb-Edu (web text) + The Stack (code)
- Fully open-source, no login required

>  **Real-world examples:** CodeLlama was built by continued pretraining of
> LLaMA-2 on code data. BioMedLM adapted GPT-2 on PubMed abstracts.
> In both cases, the general model's weights provided a much better starting
> point than random initialisation.


In [11]:
from transformers import AutoModelForCausalLM

# Load SmolLM2-360M pretrained weights from HuggingFace Hub
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

print("Pretrained model loaded:")
print_nparams(model)
print(f"\nArchitecture: {model.config.model_type}")
print(f"Layers      : {model.config.num_hidden_layers}")
print(f"Hidden size : {model.config.hidden_size}")


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Pretrained model loaded:
Total parameters: 361,821,120  (~362M)

Architecture: llama
Layers      : 32
Hidden size : 960


#### Inference on the Pretrained Model

Unlike the random model, this one was trained on billions of tokens.


In [12]:
prompt = "Large language models are trained on"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f"Prompt: '{prompt}'")
print("\nSmolLM2-360M output (pretrained — expect coherent text):")
outputs = model.generate(
    **inputs,
    streamer=streamer,
    use_cache=True,
    max_new_tokens=64,
    do_sample=False
)


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Prompt: 'Large language models are trained on'

SmolLM2-360M output (pretrained — expect coherent text):
 large corpora of text, and the training process is time-consuming.

The training process is time-consuming because it requires a large corpus of text.

The training process is time-consuming because it requires a large corpus of text.

The training process is time-consuming because it requires a


> **Observation:** The pretrained model produces fluent, coherent text on the same
> prompt that produced gibberish with random weights. This is the entire value of
> pretraining — and why continued pretraining from a good checkpoint is preferred
> over training from scratch when feasible.


In [13]:
# Free memory
del model, streamer, outputs
gc.collect()
print("Memory cleared.")


Memory cleared.


### 2c. Downscaling (Depth Pruning)

Create a **smaller model** by removing entire transformer layers from the middle
of a pretrained model, then retrain to recover quality.

**Why remove middle layers?**
- Bottom layers learn low-level syntax and token patterns
- Top layers learn high-level reasoning and task behaviour
- Middle layers are the most redundant and safest to remove

**When to use:**
- You need a faster/smaller model for inference on constrained hardware
- You want to create a student model before knowledge distillation
- You want a cheaper continued pretraining starting point

> **Depth pruning vs weight pruning:**
> - **Depth pruning** (this demo) removes entire transformer layers — coarse but simple
> - **Weight pruning** removes individual weights by magnitude — finer-grained but complex
> - **Structured pruning** removes attention heads or neurons — a middle ground
>
> Here we downscale SmolLM2-360M from **30 layers → 26 layers** by removing
> layers 13 and 14 (the two innermost middle layers).


In [15]:
from transformers import AutoConfig

# Load SmolLM2-360M as the base model to downscale
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

print("Before downscaling:")
print_nparams(model)
print(f"Number of layers: {len(model.model.layers)}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Before downscaling:
Total parameters: 361,821,120  (~362M)
Number of layers: 32


In [16]:
# Inspect layer names to understand the structure
print("Model layer names:")
for i, layer in enumerate(model.model.layers):
    print(f"  Layer {i}: {layer.__class__.__name__}")


Model layer names:
  Layer 0: LlamaDecoderLayer
  Layer 1: LlamaDecoderLayer
  Layer 2: LlamaDecoderLayer
  Layer 3: LlamaDecoderLayer
  Layer 4: LlamaDecoderLayer
  Layer 5: LlamaDecoderLayer
  Layer 6: LlamaDecoderLayer
  Layer 7: LlamaDecoderLayer
  Layer 8: LlamaDecoderLayer
  Layer 9: LlamaDecoderLayer
  Layer 10: LlamaDecoderLayer
  Layer 11: LlamaDecoderLayer
  Layer 12: LlamaDecoderLayer
  Layer 13: LlamaDecoderLayer
  Layer 14: LlamaDecoderLayer
  Layer 15: LlamaDecoderLayer
  Layer 16: LlamaDecoderLayer
  Layer 17: LlamaDecoderLayer
  Layer 18: LlamaDecoderLayer
  Layer 19: LlamaDecoderLayer
  Layer 20: LlamaDecoderLayer
  Layer 21: LlamaDecoderLayer
  Layer 22: LlamaDecoderLayer
  Layer 23: LlamaDecoderLayer
  Layer 24: LlamaDecoderLayer
  Layer 25: LlamaDecoderLayer
  Layer 26: LlamaDecoderLayer
  Layer 27: LlamaDecoderLayer
  Layer 28: LlamaDecoderLayer
  Layer 29: LlamaDecoderLayer
  Layer 30: LlamaDecoderLayer
  Layer 31: LlamaDecoderLayer


In [17]:
# Remove the two middle layers (keep bottom half + top half)
total_layers = len(model.model.layers)
keep_bottom = total_layers // 2 - 1   # 14 layers from bottom
keep_top = total_layers // 2 - 1      # 14 layers from top

layers = model.model.layers
model.model.layers = layers[:keep_bottom] + layers[-keep_top:]

# Update config to reflect new layer count
config_pruned = AutoConfig.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M",
    num_hidden_layers=len(model.model.layers),
)
model.config = config_pruned

print("After downscaling:")
print_nparams(model)
print(f"Layers remaining : {len(model.model.layers)}  (removed {total_layers - len(model.model.layers)} middle layers)")


After downscaling:
Total parameters: 342,156,480  (~342M)
Layers remaining : 30  (removed 2 middle layers)


#### Inference on the Downscaled Model

The downscaled model still retains pretrained weights in all surviving layers.
Output quality degrades slightly but remains far better than random initialisation.
A short continued pretraining run would restore most of the original quality:


In [18]:
prompt = "Large language models are trained on"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f"Prompt: '{prompt}'")
print("\nDownscaled model output:")
outputs = model.generate(
    **inputs,
    streamer=streamer,
    use_cache=True,
    max_new_tokens=64,
    do_sample=False
)


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Prompt: 'Large language models are trained on'

Downscaled model output:
 a large corpus of text, and the model is trained to predict the next word in the text.

The model is trained on a large corpus of text, and the model is trained to predict the next word in the text.

The model is trained on a large corpus of text, and the model is


## 3. Save the Model to Google Drive

Save the downscaled model as the starting checkpoint for the pretraining training loop.

> The naming convention `SmolLM2-26L-pruned-init` reflects:
> - **SmolLM2** → base model family
> - **26L** → 26 layers after pruning
> - **pruned** → depth pruning applied
> - **init** → initialised, pretraining not yet run


In [19]:
import os

save_path = f"{data_dir}/SmolLM2-26L-pruned-init"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")
print("\nSaved files:")
for f in os.listdir(save_path):
    print(f"  - {f}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/Colab Notebooks/data/SmolLM2-26L-pruned-init

Saved files:
  - config.json
  - generation_config.json
  - model.safetensors
  - tokenizer_config.json
  - tokenizer.json


In [20]:
# Free memory
del model, streamer, outputs
gc.collect()
print("Memory cleared.")


Memory cleared.


| Concept | Summary |
|---|---|
| `LlamaConfig` | Defines architecture hyperparameters independently of weights |
| Random init | Truncated normal (mean=0, std=0.02) — baseline for training from scratch |
| Continued pretraining | Load pretrained weights, resume training on new domain data |
| Depth pruning | Remove middle transformer layers to create a smaller model |
| Weight vs depth pruning | Weight pruning = individual weights; Depth pruning = entire layers |
| `print_nparams()` | Essential sanity check after any architectural modification |
| bfloat16 | Half-precision dtype — halves memory footprint |
| `use_cache=False` | Required when gradient checkpointing is enabled during training |